# B1 : Prompt craft I : intention, contexte, format

**Objectif du bloc :** écrire des prompts qui marchent du premier coup sur des tâches réelles, et savoir *pourquoi* ils marchent.

Ce notebook **relit les résultats pré-calculés** (`notebooks/b1_results/`). Les expériences ont été réellement exécutées via l'API
(`forge.clients.complete`, journal local dans `data/calls.jsonl`) par les scripts :

| Script | Expérience | Sortie |
|---|---|---|
| `scripts/b1_ab.py` | A/B (a) brut · (b) 6Q · (c) brut + contexte : 5 tâches × 2 modèles × 2 répétitions | `ab_runs.jsonl`, `ab_scores.csv` |
| `scripts/b1_ablation.py` | Ablation du prompt 6Q (un élément retiré à la fois) : 2 tâches × 6 variantes × 2 modèles × 4 rép. | `ablation_*.{jsonl,csv}` |
| `scripts/b1_json.py` | JSON + Pydantic, 20 runs par condition | `json_*.{jsonl,csv}` |
| `scripts/b1_library_smoke.py` | Test réel des 5 gabarits annexes | `library_*.{jsonl,csv}` |
| `scripts/b1_rejudge.py` | Complète les notes du juge Gemini perdues sur quota 429 | (réécrit les fichiers ci-dessus) |

**Modèles testés :** `gpt-4o-mini`, `anthropic/claude-haiku-4-5` (température 0,7, `max_tokens` 1600).
**Juges :** `gemini/gemini-3.5-flash-lite` et `xai/grok-3-mini` (température 0), d'autres familles que les modèles testés, pour limiter l'auto-préférence.
Grille courte (1–5) : clarté, complétude, respect du format, utilité ; le juge reçoit le **besoin réel** (contexte complet), même quand le prompt évalué ne le contenait pas.
Score = moyenne des 4 critères, puis moyenne des 2 juges.

Fixtures (fictives mais riches, commitées) : `forge/prompts/_fixtures/`. Pour tout relancer (~0,7 $) :
```bash
source .venv/bin/activate && cd scripts
python b1_ab.py && python b1_ablation.py && python b1_json.py && python b1_library_smoke.py && python b1_rejudge.py
```

In [1]:
import pathlib, sys
import pandas as pd
pd.set_option("display.precision", 2)
ROOT = pathlib.Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT)); sys.path.insert(0, str(ROOT / "scripts"))
R = ROOT / "notebooks" / "b1_results"
ab = pd.read_csv(R / "ab_scores.csv")
abl = pd.read_csv(R / "ablation_scores.csv")
js = pd.read_csv(R / "json_scores.csv")
lib = pd.read_csv(R / "library_scores.csv")
G, X = "score_gemini-3.5-flash-lite", "score_grok-3-mini"
print(len(ab), "runs A/B ;", len(abl), "runs d'ablation ;", len(js), "runs JSON ;", len(lib), "runs bibliothèque")
print("Notes manquantes :", int(ab[[G, X]].isna().sum().sum() + abl[[G, X]].isna().sum().sum()))

60 runs A/B ; 96 runs d'ablation ; 40 runs JSON ; 20 runs bibliothèque
Notes manquantes : 0


## 1. A/B : (a) brut · (b) « 6 questions » · (c) brut + contexte complet

Les 5 tâches : résumer un papier (`t1`), refactorer une fonction (`t2`), comparer LiteLLM et OpenRouter (`t3`),
rédiger un courriel d'incident (`t4`), extraire des données d'un rapport (`t5`).

- **(a)** une ligne tapée en 30 s, typos comprises (+ l'artefact quand la tâche en a un : papier, code, rapport) ;
- **(b)** gabarit `forge/prompts/<id>.md` : Rôle · Tâche · Contexte · Contraintes · Format · Critère (+ un exemple) ;
- **(c)** le prompt (a) + les notes de contexte collées telles quelles, sans structure.

In [2]:
from b1_common import build_prompt, TASKS
pd.DataFrame({v: {t: len(build_prompt(t, v)) for t in TASKS} for v in "abc"}).rename_axis("longueur du prompt (car.)")

,a,b,c
longueur du prompt (car.),,,
t1_resume,3260,4539,3807
t2_refactor,890,2403,1601
t3_comparer,48,3292,2168
t4_email,150,1844,1393
t5_extraire,1763,3543,2449


In [3]:
print("Moyenne par modèle et variante (2 juges)")
display(ab.pivot_table(index="model", columns="variant", values="score", margins=True, margins_name="tous"))
print("Moyenne par tâche et variante")
display(ab.pivot_table(index="task", columns="variant", values="score"))

Moyenne par modèle et variante (2 juges)


variant,a,b,c,tous
model,,,,
anthropic/claude-haiku-4-5,2.81,4.66,4.71,4.06
gpt-4o-mini,2.33,4.61,4.50,3.81
tous,2.57,4.64,4.61,3.94


Moyenne par tâche et variante


variant,a,b,c
task,,,
t1_resume,2.81,4.19,3.84
t2_refactor,3.06,4.12,4.25
t3_comparer,2.34,5.00,5.00
t4_email,2.25,4.88,4.94
t5_extraire,2.38,5.00,5.00


In [4]:
pt = ab.pivot_table(index=["task", "model"], columns="variant", values="score")
pt["b-a"] = pt.b - pt.a; pt["c-b"] = pt.c - pt.b
display(pt)
print(f"b > a dans {(pt.b > pt.a).sum()}/10 cellules ; c ≥ b dans {(pt.c >= pt.b).sum()}/10 cellules")
print("Écart-type intra-variante :", ab.groupby("variant").score.std().round(2).to_dict())

variant                                    a     b     c   b-a   c-b
task        model                                                   
t1_resume   anthropic/claude-haiku-4-5  3.31  4.56  4.62  1.25  0.06
            gpt-4o-mini                 2.31  3.81  3.06  1.50 -0.75
t2_refactor anthropic/claude-haiku-4-5  3.19  3.75  3.94  0.56  0.19
            gpt-4o-mini                 2.94  4.50  4.56  1.56  0.06
t3_comparer anthropic/claude-haiku-4-5  2.94  5.00  5.00  2.06  0.00
            gpt-4o-mini                 1.75  5.00  5.00  3.25  0.00
t4_email    anthropic/claude-haiku-4-5  2.19  5.00  5.00  2.81  0.00
            gpt-4o-mini                 2.31  4.75  4.88  2.44  0.12
t5_extraire anthropic/claude-haiku-4-5  2.44  5.00  5.00  2.56  0.00
            gpt-4o-mini                 2.31  5.00  5.00  2.69  0.00

b > a dans 10/10 cellules ; c ≥ b dans 9/10 cellules
Écart-type intra-variante : {'a': 0.5, 'b': 0.5, 'c': 0.64}


In [5]:
print("Robustesse : chaque juge séparément")
display(ab.groupby("variant")[[G, X]].mean())
print("Corrélation entre juges :", round(ab[[G, X]].corr().iloc[0, 1], 2))
print("Critères (moyenne des 2 juges)")
display(ab.groupby("variant")[["clarte", "completude", "format", "utilite", "words"]].mean())

Robustesse : chaque juge séparément


,score_gemini-3.5-flash-lite,score_grok-3-mini
variant,,
a,2.92,2.21
b,4.80,4.47
c,4.75,4.46


Corrélation entre juges : 0.9
Critères (moyenne des 2 juges)


,clarte,completude,format,utilite,words
variant,,,,,
a,4.20,2.08,2.10,1.90,219.75
b,4.85,4.70,4.47,4.53,265.55
c,4.88,4.55,4.47,4.53,288.95


### Conclusion A/B

| | (a) brut | (b) 6Q | (c) brut + contexte |
|---|---|---|---|
| gpt-4o-mini | 2,33 | **4,61** | 4,50 |
| claude-haiku-4-5 | 2,81 | 4,66 | **4,71** |
| **Les deux** | **2,57** | **4,64** | **4,61** |

- **(b) > (a) : confirmé, largement** (+2,1 points, 10 cellules sur 10, et pour les deux juges séparément). Le gain vient de la **complétude** (2,1 → 4,7), du **format** (2,1 → 4,5) et de l'**utilité** (1,9 → 4,5), bien plus que de la clarté (4,2 → 4,9) : le brut est lisible mais il répond à côté du besoin.
- **(c) ≥ (b) : confirmé cellule par cellule (9/10), égalité en moyenne** (4,61 vs 4,64, écart dans le bruit, σ ≈ 0,5). Coller le contexte en vrac donne le même résultat que le prompt 6Q soigné : **le contexte fait le travail, pas la structure.**
- L'exception est instructive : `t1` × gpt-4o-mini, (c) 3,06 < (b) 3,81. Quand le besoin de format est noyé dans des notes en vrac (« 200 mots, en puces, avis clair »), gpt-4o-mini l'ignore et paraphrase l'abstract ; la section `Format` explicite le corrige. Haiku n'en a pas besoin (4,62 vs 4,56).
- Les tâches `t3`, `t4` et `t5` saturent à 5/5 en (b) et (c) : pour les départager, il faudra des critères objectifs (B10).
- Les typos du brut ne gênent pas les modèles ; c'est l'absence de contexte qui coûte.

## 2. Ablation : retirer un élément à la fois du prompt 6Q

Tâches retenues : `t1_resume` (seule tâche avec de la marge au juge, plus des contraintes mesurables : ≤ 200 mots, copie de l'exemple) et
`t5_extraire` (le meilleur prompt, 5/5, avec des métriques objectives : JSON strict, exactitude des champs).
4 répétitions × 2 modèles par variante. `copie_exemple` = similarité max (difflib) entre la phrase d'exemple et une ligne de la sortie.

In [6]:
t1 = abl[abl.task == "t1_resume"]
display(t1.pivot_table(index="retire", columns="model", values="score"))
display(t1.groupby(["retire", "model"])[["format", "words", "respect_200_mots", "copie_exemple"]].mean())
print("σ du score par cellule :")
display(t1.groupby(["retire", "model"]).score.std().unstack())

model,anthropic/claude-haiku-4-5,gpt-4o-mini
retire,,
(complet),4.72,3.97
Contraintes,4.66,3.78
Critère de réussite,4.69,3.22
Exemple,4.94,3.38
Format,4.88,3.03
Rôle,4.69,3.72


format   words  \
retire              model                                        
(complet)           anthropic/claude-haiku-4-5    4.12  252.50   
                    gpt-4o-mini                   3.12  193.25   
Contraintes         anthropic/claude-haiku-4-5    4.12  340.50   
                    gpt-4o-mini                   2.62  244.75   
Critère de réussite anthropic/claude-haiku-4-5    4.12  252.50   
                    gpt-4o-mini                   2.25  212.75   
Exemple             anthropic/claude-haiku-4-5    4.75  243.00   
                    gpt-4o-mini                   2.38  207.75   
Format              anthropic/claude-haiku-4-5    4.50  191.00   
                    gpt-4o-mini                   2.00  144.25   
Rôle                anthropic/claude-haiku-4-5    4.12  238.00   
                    gpt-4o-mini                   2.75  205.00   

                                               respect_200_mots  copie_exemple  
retire              model                                                       
(complet)           anthropic/claude-haiku-4-5              0.0           0.33  
                    gpt-4o-mini                             1.0           0.76  
Contraintes         anthropic/claude-haiku-4-5              0.0           0.33  
                    gpt-4o-mini                             0.0           0.61  
Critère de réussite anthropic/claude-haiku-4-5              0.0           0.35  
                    gpt-4o-mini                             0.0           0.61  
Exemple             anthropic/claude-haiku-4-5              0.0           0.33  
                    gpt-4o-mini                             0.0           0.38  
Format              anthropic/claude-haiku-4-5             0.75           0.38  
                    gpt-4o-mini                             1.0           0.66  
Rôle                anthropic/claude-haiku-4-5              0.0           0.35  
                    gpt-4o-mini                            0.25           0.67

σ du score par cellule :


model,anthropic/claude-haiku-4-5,gpt-4o-mini
retire,,
(complet),0.16,0.31
Contraintes,0.26,0.47
Critère de réussite,0.24,0.41
Exemple,0.07,0.23
Format,0.14,0.12
Rôle,0.16,0.40


In [7]:
t5 = abl[abl.task == "t5_extraire"]
display(t5.groupby(["retire", "model"])[["score", "json_strict", "json_tolerant", "field_acc", "words"]].mean())

score json_strict  \
retire              model                                           
(complet)           anthropic/claude-haiku-4-5   5.00         0.0   
                    gpt-4o-mini                  5.00         1.0   
Contraintes         anthropic/claude-haiku-4-5   5.00         0.0   
                    gpt-4o-mini                  5.00         1.0   
Critère de réussite anthropic/claude-haiku-4-5   5.00         0.0   
                    gpt-4o-mini                  5.00         1.0   
Exemple             anthropic/claude-haiku-4-5   5.00         0.0   
                    gpt-4o-mini                  5.00         1.0   
Format              anthropic/claude-haiku-4-5   5.00         0.0   
                    gpt-4o-mini                  5.00        0.25   
Rôle                anthropic/claude-haiku-4-5   5.00         0.0   
                    gpt-4o-mini                  4.53         1.0   

                                               json_tolerant  field_acc  words  
retire              model                                                       
(complet)           anthropic/claude-haiku-4-5           1.0       1.00  110.0  
                    gpt-4o-mini                          1.0       1.00   36.0  
Contraintes         anthropic/claude-haiku-4-5           1.0       1.00  110.0  
                    gpt-4o-mini                          1.0       1.00   39.0  
Critère de réussite anthropic/claude-haiku-4-5           1.0       1.00  110.0  
                    gpt-4o-mini                          1.0       1.00   36.0  
Exemple             anthropic/claude-haiku-4-5           1.0       1.00  122.0  
                    gpt-4o-mini                          1.0       1.00  120.0  
Format              anthropic/claude-haiku-4-5           1.0       1.00  122.0  
                    gpt-4o-mini                          1.0       1.00  121.5  
Rôle                anthropic/claude-haiku-4-5           1.0       1.00  110.0  
                    gpt-4o-mini                          1.0       0.92   38.0

### Conclusion ablation

**Éléments prouvés inutiles (sur `t5_extraire`, le meilleur prompt) : `Critère de réussite` et `Contraintes`.**
Les retirer ne change **aucune** métrique sur 8 runs : score du juge 5,0, JSON strict identique (gpt 4/4, Haiku 0/4), exactitude des champs 100 %,
même longueur de sortie. Le schéma JSON (dans `Contexte`) porte déjà les règles des champs (« null si pas de date précise », énumération des statuts) :
les consignes de `Contraintes` faisaient doublon.

Ce qui compte vraiment :
- **`Format` est indispensable pour le JSON strict** : sans lui, gpt-4o-mini passe de 4/4 à 1/4 JSON parsable sans nettoyage (il rajoute un bloc ```json).
- **`Exemple` fixe la forme, pas le fond** : sur `t5`, l'exemple compact fait sortir gpt-4o-mini en JSON sur une ligne (36 mots contre 120 sans). Sur `t1`, il est **contre-productif** : gpt-4o-mini recopie la phrase d'exemple (similarité 0,76 → 0,38 sans exemple) et Haiku est **meilleur sans** (4,72 → 4,94, format 4,1 → 4,75). Mais gpt-4o-mini perd 0,6 point sans exemple : un exemple unique est un mauvais compromis → v2 avec 2–3 exemples variés (théorie du B1).
- **`Rôle` : effet faible et dans le bruit** sur `t1` (−0,03 pour Haiku, −0,25 pour gpt avec σ ≈ 0,4) ; sur `t5`, sans rôle, gpt-4o-mini fait des erreurs de champs (exactitude moyenne 0,92 au lieu de 1,0). Candidat « inutile » sur Haiku, à confirmer avec plus de runs.
- **`Format` et `Critère` comptent pour gpt-4o-mini sur `t1`** (−0,9 et −0,75), quasi pas pour Haiku : **l'utilité d'un élément dépend du modèle.**
- La contrainte de longueur « 200 mots » n'est **jamais** respectée par Haiku avec le prompt complet (0/8, ≈ 250 mots) : une limite en mots est un mauvais levier pour ce modèle.

## 3. Format JSON + Pydantic (tâche `t5_extraire`)

Schéma `RapportProjets` (Pydantic v2, `extra="forbid"`, `statut: Literal[...]`, `date_prochain_jalon: date | None`, montants `int`).
**Valide = `RapportProjets.model_validate_json(sortie_brute)`** sans aucun nettoyage. 20 runs par condition (10 gpt-4o-mini + 10 claude-haiku-4-5), température 0,7.

- `v1` : gabarit `extraire-json` seul (« pas de bloc de code Markdown ») ;
- `v1+prefill` : même gabarit ; pour Anthropic, le tour assistant est pré-rempli par `{` (technique documentée par Anthropic ; la sortie validée = `{` + texte).

In [8]:
import inspect, b1_common
print(inspect.getsource(b1_common.Projet)); print(inspect.getsource(b1_common.RapportProjets))

class Projet(BaseModel):
    model_config = ConfigDict(extra="forbid")
    nom: str
    responsable: str
    statut: Literal["en_cours", "en_pause", "termine"]
    budget_alloue_eur: int = Field(ge=0)
    budget_consomme_eur: int = Field(ge=0)
    etp: float = Field(ge=0)
    prochain_jalon: str | None
    date_prochain_jalon: date | None
    risque_principal: str | None

class RapportProjets(BaseModel):
    model_config = ConfigDict(extra="forbid")
    projets: list[Projet] = Field(min_length=1)



In [9]:
summary = js.groupby("condition").agg(runs=("strict_valid", "size"), json_strict=("strict_valid", "sum"),
                                     json_tolerant=("tolerant_valid", "sum"), exactitude_champs=("field_acc", "mean"))
summary["% strict"] = 100 * summary.json_strict / summary.runs
display(summary)
display(js.groupby(["condition", "model"])[["strict_valid", "tolerant_valid", "field_acc"]].mean())
print("Erreurs v1 (Haiku) :", js[(js.condition == "v1") & ~js.strict_valid].validation_error.str[:90].unique()[:2])

,runs,json_strict,json_tolerant,exactitude_champs,% strict
condition,,,,,
v1,20,10,20,1.0,50.0
v1+prefill,20,20,20,1.0,100.0


strict_valid  tolerant_valid  field_acc
condition  model                                                              
v1         anthropic/claude-haiku-4-5           0.0             1.0        1.0
           gpt-4o-mini                          1.0             1.0        1.0
v1+prefill anthropic/claude-haiku-4-5           1.0             1.0        1.0
           gpt-4o-mini                          1.0             1.0        1.0

Erreurs v1 (Haiku) : <StringArray>
['1 validation error for RapportProjets']
Length: 1, dtype: str


### Conclusion JSON

- **`v1+prefill` : 20/20 JSON strict valide (100 %, cible ≥ 95 % atteinte)**, exactitude des champs 100 % vs vérité terrain (4 projets × 7 champs).
- `v1` seul : 10/20 (50 %). gpt-4o-mini 10/10 ; **claude-haiku-4-5 0/10 : il entoure toujours le JSON d'un bloc ```json malgré la consigne explicite.** Le contenu est parfait (20/20 après retrait du bloc).
- Leçon : la consigne « pas de bloc de code » ne suffit pas pour Haiku ; le **prefill `{`** (ou un parseur tolérant, ou le mode JSON natif) règle le problème à coût nul. En A/B, le brut (a) donnait 0/4 JSON exploitable : texte libre ou clés inventées.

## 4. Bibliothèque v1 : 10 gabarits `forge/prompts/*.md`

Les 5 gabarits de l'A/B sont les variantes (b). Les 5 gabarits annexes (A.3, A.4, A.5, A.6 adaptés + résumé de thread GitHub) ont été testés
en réel (2 modèles × 2 runs, même double juge).

In [10]:
from forge.prompts import list_prompts
pd.DataFrame([{"id": p.id, "version": p.meta["version"], "variables": ", ".join(p.variables),
               "score_moyen": p.meta["score_moyen"], "sections": " · ".join(p.section_names())} for p in list_prompts()])

,id,version,variables,score_moyen,sections
0,ameliorer-prompt-court,1,"prompt, echecs, reussites",4.72,Rôle · Tâche · Contexte · Contraintes · Format
1,brief-mission-court,1,"demande, ressources, budget",4.91,Rôle · Tâche · Contexte · Contraintes · Format...
2,comparer-outils,1,"outil_a, outil_b, notes, decideur",5.00,Rôle · Tâche · Contexte · Contraintes · Format...
3,critique-reponse,1,"brouillon, grille, objectif",4.03,Rôle · Tâche · Contexte · Contraintes · Format...
4,email-technique,1,"expediteur, destinataire, faits, demandes, ton...",4.88,Rôle · Tâche · Contexte · Contraintes · Format...
5,extraire-json,1,"texte, schema, consignes_champs",5.00,Rôle · Tâche · Contexte · Contraintes · Format...
6,refactor-fonction,1,"code, contexte, contraintes_equipe",4.12,Rôle · Tâche · Contexte · Contraintes · Format...
7,resume-papier,1,"texte, public, objectif, longueur",4.19,Rôle · Tâche · Contexte · Contraintes · Format...
8,resume-thread-github,1,"thread, lecteur",4.59,Rôle · Tâche · Contraintes · Format · Critère ...
9,scoring-pertinence,1,"interets, item",5.00,Tâche · Contexte · Contraintes · Format


In [11]:
display(lib.pivot_table(index="template", columns="model", values="score", margins=True, margins_name="moyenne"))
print("JSON valide scoring-pertinence :", int(lib.json_valid.sum()), "/", int(lib.json_valid.notna().sum()))
print("Consigne injectée reprise dans le résumé de thread :", lib.dropna(subset=["injection_reprise"]).groupby("model").injection_reprise.sum().to_dict())

model,anthropic/claude-haiku-4-5,gpt-4o-mini,moyenne
template,,,
ameliorer-prompt-court,5.00,4.44,4.72
brief-mission-court,5.00,4.81,4.91
critique-reponse,5.00,3.06,4.03
resume-thread-github,4.69,4.50,4.59
scoring-pertinence,5.00,5.00,5.00
moyenne,4.94,4.36,4.65


JSON valide scoring-pertinence : 4 / 4
Consigne injectée reprise dans le résumé de thread : {'anthropic/claude-haiku-4-5': 0, 'gpt-4o-mini': 2}


## 5. Synthèse et limites

**Hypothèse (c) ≥ (b) > (a) : confirmée.** b > a très nettement (+2,1) ; c ≈ b (égalité, c ≥ b dans 9 cellules sur 10).
**Élément inutile :** `Critère de réussite` (et `Contraintes`, redondantes avec le schéma) sur l'extraction ; `Exemple` unique nuisible sur le résumé pour Haiku.
**JSON :** 20/20 (100 %) avec prefill, 10/20 sans.

Limites (à garder en tête avant d'en faire des règles) :
- Petits échantillons (2 rép. par cellule en A/B, 4 en ablation) : seuls les écarts > ~0,5 sont lisibles.
- Juges LLM : Gemini note ~0,3–0,7 plus haut que Grok, mais les deux classent les variantes pareil (r = 0,9). Le juge voit le besoin réel, ce qui pénalise (a) *par construction* : c'est voulu (on mesure l'utilité pour l'utilisateur).
- Effet plafond sur `t3`/`t4`/`t5` : il faudra des cas plus durs et des critères objectifs (B10).
- Fixtures fictives écrites pour l'exercice : les vraies tâches de la semaine seront plus bruitées.